###1. Objective

To tune the shortlisted WB+CLAHE + YOLOv8n candidates using a controlled hyperparameter search,compare validation performance,and save the best validated configuration for each shortlisted dataset.

###2. Shortlisted Candidates

From Day 18,we selected:

Aquatic Plant + WB+CLAHE + YOLOv8n
Well + WB+CLAHE + YOLOv8n

The original YOLOv8n models remain the baseline for comparison.

###3. Experimental Strategy

We will change one parameter at a time while keeping the other settings fixed.

Parameter 1 — Learning Rate

We will test:

lr0=0.001
lr0=0.0005

Batch size will remain 16 while comparing learning rates.

Parameter 2 — Batch Size

After selecting the better learning rate,we will test:

batch=16
batch=32

The selected learning rate will remain fixed during this comparison

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

SOURCE="/content/drive/MyDrive/Dataset_V1"
ENHANCED="/content/drive/MyDrive/Dataset_V1_Enhanced"

def white_balance(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    avg_a=np.mean(a)
    avg_b=np.mean(b)
    a=np.clip(a-(avg_a-128)*l/255,0,255).astype(np.uint8)
    b=np.clip(b-(avg_b-128)*l/255,0,255).astype(np.uint8)
    lab=cv2.merge((l,a,b))
    return cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)

def apply_clahe(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l=clahe.apply(l)
    enhanced=cv2.merge((l,a,b))
    return cv2.cvtColor(enhanced,cv2.COLOR_LAB2BGR)

def enhance_image(img):
    img=white_balance(img)
    img=apply_clahe(img)
    return img

tasks=[
    ("Aquatic Plant.v2i.yolov8","valid"),
    ("Aquatic Plant.v2i.yolov8","test"),
    ("well.v8i.yolov8","valid"),
    ("well.v8i.yolov8","test")
]

for dataset,split in tasks:
    src_images=os.path.join(SOURCE,dataset,split,"images")
    dst_images=os.path.join(ENHANCED,dataset,split,"images")
    src_labels=os.path.join(SOURCE,dataset,split,"labels")
    dst_labels=os.path.join(ENHANCED,dataset,split,"labels")

    os.makedirs(dst_images,exist_ok=True)
    os.makedirs(dst_labels,exist_ok=True)

    files=[f for f in os.listdir(src_images)
           if f.lower().endswith((".jpg",".jpeg",".png",".webp"))]

    print(f"\n{dataset} - {split}: {len(files)} images")

    for file in tqdm(files):
        src=os.path.join(src_images,file)
        dst=os.path.join(dst_images,file)

        img=cv2.imread(src)

        if img is None:
            print("Could not read:",file)
            continue

        enhanced=enhance_image(img)
        cv2.imwrite(dst,enhanced)

    label_files=[f for f in os.listdir(src_labels)
                 if f.lower().endswith(".txt")]

    for file in label_files:
        src=os.path.join(src_labels,file)
        dst=os.path.join(dst_labels,file)

        if not os.path.exists(dst):
            with open(src,"r") as f:
                content=f.read()

            with open(dst,"w") as f:
                f.write(content)

    print("Completed:",dataset,split)

print("\nAll missing enhanced splits completed.")


Aquatic Plant.v2i.yolov8 - valid: 179 images


100%|██████████| 179/179 [00:12<00:00, 14.74it/s]


Completed: Aquatic Plant.v2i.yolov8 valid

Aquatic Plant.v2i.yolov8 - test: 89 images


100%|██████████| 89/89 [01:10<00:00,  1.25it/s]


Completed: Aquatic Plant.v2i.yolov8 test

well.v8i.yolov8 - valid: 191 images


100%|██████████| 191/191 [00:10<00:00, 18.26it/s]


Completed: well.v8i.yolov8 valid

well.v8i.yolov8 - test: 184 images


100%|██████████| 184/184 [00:12<00:00, 14.48it/s]


Completed: well.v8i.yolov8 test

All missing enhanced splits completed.


In [ ]:
import os
base="/content/drive/MyDrive/Dataset_V1_Enhanced"
datasets={
    "DIATAquarium.v4i.yolov8":["train","valid"],
    "Aquatic Plant.v2i.yolov8":["train","valid","test"],
    "well.v8i.yolov8":["train","valid","test"]
}
for dataset,splits in datasets.items():
    print("\n",dataset)
    for split in splits:
        img_path=os.path.join(base,dataset,split,"images")
        label_path=os.path.join(base,dataset,split,"labels")
        images=[f for f in os.listdir(img_path)
                if f.lower().endswith((".jpg",".jpeg",".png",".webp"))]
        labels=[f for f in os.listdir(label_path)
                if f.lower().endswith(".txt")]

        print(f"{split}: {len(images)} images | {len(labels)} labels")


 DIATAquarium.v4i.yolov8
train: 8121 images | 8121 labels
valid: 773 images | 773 labels

 Aquatic Plant.v2i.yolov8
train: 1892 images | 1879 labels
valid: 179 images | 179 labels
test: 89 images | 89 labels

 well.v8i.yolov8
train: 3795 images | 3780 labels
valid: 191 images | 191 labels
test: 184 images | 184 labels


In [ ]:
import os
import shutil
import time
source="/content/drive/MyDrive/Dataset_V1_Enhanced"
destination="/content/Dataset_V1_Enhanced"
if os.path.exists(destination):
    shutil.rmtree(destination)
start=time.time()
shutil.copytree(source,destination)
elapsed=time.time()-start
print("Enhanced dataset copied successfully.")
print(f"Time taken: {elapsed/60:.2f} minutes")
print("Location:",destination)

Enhanced dataset copied successfully.
Time taken: 11.10 minutes
Location: /content/Dataset_V1_Enhanced


In [ ]:
import os
AP_BASE="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8"
AP_YAML="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml"
with open(AP_YAML,"w") as f:
    f.write(f"""path: {AP_BASE}
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant
""")
print("H01 YAML created:")
print(AP_YAML)
print("\nContents:")
with open(AP_YAML,"r") as f:
    print(f.read())


H01 YAML created:
/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml

Contents:
path: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant



In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 7.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import time
H01_YAML="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H01_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.001,
    device=0,
    project="/content/Day19_YOLO_Tuning",
    name="H01_AP_lr001_batch16"
)
training_time=time.time()-start
print(f"\nH01 training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, 

In [ ]:
from ultralytics import YOLO
import time
H01_YAML="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H01_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.001,
    optimizer="AdamW",
    device=0,
    project="/content/Day19_YOLO_Tuning",
    name="H01_AP_lr001_batch16_fixed",
    exist_ok=True
)
training_time=time.time()-start
print("\nH01 corrected training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=H01_AP_lr001_

In [ ]:
from ultralytics import YOLO
import time
H02_YAML="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H02_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.0005,
    optimizer="AdamW",
    device=0,
    project="/content/Day19_YOLO_Tuning",
    name="H02_AP_lr0005_batch16",
    exist_ok=True
)
training_time=time.time()-start
print("\nH02 training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=H02_AP_lr000

In [ ]:
from ultralytics import YOLO
import time
H03_YAML="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H03_YAML,
    epochs=30,
    imgsz=640,
    batch=32,
    lr0=0.0005,
    optimizer="AdamW",
    device=0,
    project="/content/Day19_YOLO_Tuning",
    name="H03_AP_lr0005_batch32",
    exist_ok=True
)
training_time=time.time()-start
print("\nH03 training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=H03_AP_lr000

In [ ]:
import pandas as pd
data=[
["H01",0.001,16,94.5,95.4,98.2,71.3],
["H02",0.0005,16,94.5,94.9,98.2,71.4],
["H03",0.0005,32,95.4,96.1,98.5,71.5]
]
df=pd.DataFrame(data,columns=["Experiment","Learning_Rate","Batch_Size","Precision","Recall","mAP50","mAP50_95"])
path="/content/drive/MyDrive/Aquatic_Plant_Day19_Summary.csv"
df.to_csv(path,index=False)
print("Summary saved:",path)

Summary saved: /content/drive/MyDrive/Aquatic_Plant_Day19_Summary.csv


In [ ]:
import os
WELL_BASE="/content/Dataset_V1_Enhanced/well.v8i.yolov8"
WELL_YAML="/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml"
with open(WELL_YAML,"w") as f:
    f.write(f"""path: {WELL_BASE}
train: train/images
val: valid/images
test: test/images
nc: 4
names:
  0: Inlet-pipe
  1: fishes
  2: school-of-fish
  3: stone
""")
print("H04 YAML created:")
print(WELL_YAML)
print("\nContents:")
with open(WELL_YAML,"r") as f:
    print(f.read())

H04 YAML created:
/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml

Contents:
path: /content/Dataset_V1_Enhanced/well.v8i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 4
names:
  0: Inlet-pipe
  1: fishes
  2: school-of-fish
  3: stone



In [ ]:
from ultralytics import YOLO
import time
H04_YAML="/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H04_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.001,
    optimizer="AdamW",
    device=0,
    project="/content/Day19_YOLO_Tuning",
    name="H04_Well_lr001_batch16",
    exist_ok=True
)
training_time=time.time()-start
print("\nH04 training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=tor

In [ ]:
import pandas as pd
path="/content/drive/MyDrive/Aquatic_Plant_Day19_Summary.csv"
df=pd.read_csv(path)
df.insert(1,"Dataset",["Aquatic Plant","Aquatic Plant","Aquatic Plant"])
new_row=pd.DataFrame([[
    "H04","Well",0.001,16,35.7,40.1,36.8,16.8
]],columns=df.columns)
df=pd.concat([df,new_row],ignore_index=True)
df.to_csv(path,index=False)
print(df)
print("\nSaved successfully to:",path)

  Experiment        Dataset  Learning_Rate  Batch_Size  Precision  Recall  \
0        H01  Aquatic Plant         0.0010          16       94.5    95.4   
1        H02  Aquatic Plant         0.0005          16       94.5    94.9   
2        H03  Aquatic Plant         0.0005          32       95.4    96.1   
3        H04           Well         0.0010          16       35.7    40.1   

   mAP50  mAP50_95  
0   98.2      71.3  
1   98.2      71.4  
2   98.5      71.5  
3   36.8      16.8  

Saved successfully to: /content/drive/MyDrive/Aquatic_Plant_Day19_Summary.csv


In [ ]:
from ultralytics import YOLO
import time
H05_YAML="/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H05_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.0005,
    optimizer="AdamW",
    device=0,
    project="/content/Day19_YOLO_Tuning",
    name="H05_Well_lr0005_batch16",
    exist_ok=True
)
training_time=time.time()-start
print("\nH05 training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=H05_Well_lr0005_batch

In [ ]:
import pandas as pd
path="/content/drive/MyDrive/Aquatic_Plant_Day19_Summary.csv"
df=pd.read_csv(path)
new_row=pd.DataFrame([[
    "H05","Well",0.0005,16,43.6,33.5,37.2,16.4
]],columns=df.columns)
df=pd.concat([df,new_row],ignore_index=True)
df.to_csv(path,index=False)
print(df)
print("\nSaved successfully to:",path)

  Experiment        Dataset  Learning_Rate  Batch_Size  Precision  Recall  \
0        H01  Aquatic Plant         0.0010          16       94.5    95.4   
1        H02  Aquatic Plant         0.0005          16       94.5    94.9   
2        H03  Aquatic Plant         0.0005          32       95.4    96.1   
3        H04           Well         0.0010          16       35.7    40.1   
4        H05           Well         0.0005          16       43.6    33.5   

   mAP50  mAP50_95  
0   98.2      71.3  
1   98.2      71.4  
2   98.5      71.5  
3   36.8      16.8  
4   37.2      16.4  

Saved successfully to: /content/drive/MyDrive/Aquatic_Plant_Day19_Summary.csv


In [ ]:
from ultralytics import YOLO
import time
H06_YAML="/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H06_YAML,
    epochs=30,
    imgsz=640,
    batch=32,
    lr0=0.001,
    optimizer="AdamW",
    device=0,
    project="/content/Day19_YOLO_Tuning",
    name="H06_Well_lr001_batch32",
    exist_ok=True
)
training_time=time.time()-start
print("\nH06 training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=H06_Well_lr001_batch32

In [ ]:
import pandas as pd
path="/content/drive/MyDrive/Aquatic_Plant_Day19_Summary.csv"
df=pd.read_csv(path)
new_row=pd.DataFrame([[
    "H06","Well",0.001,32,34.4,34.5,35.3,16.1
]],columns=df.columns)
df=pd.concat([df,new_row],ignore_index=True)
df.to_csv(path,index=False)
print(df)
print("\nSaved successfully to:",path)

  Experiment        Dataset  Learning_Rate  Batch_Size  Precision  Recall  \
0        H01  Aquatic Plant         0.0010          16       94.5    95.4   
1        H02  Aquatic Plant         0.0005          16       94.5    94.9   
2        H03  Aquatic Plant         0.0005          32       95.4    96.1   
3        H04           Well         0.0010          16       35.7    40.1   
4        H05           Well         0.0005          16       43.6    33.5   
5        H06           Well         0.0010          32       34.4    34.5   

   mAP50  mAP50_95  
0   98.2      71.3  
1   98.2      71.4  
2   98.5      71.5  
3   36.8      16.8  
4   37.2      16.4  
5   35.3      16.1  

Saved successfully to: /content/drive/MyDrive/Aquatic_Plant_Day19_Summary.csv


# These are for day 22 i forgot to save so i am doing again .

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

h03="/content/Day19_YOLO_Tuning/H03_AP_lr0005_batch32/weights/best.pt"
h04="/content/Day19_YOLO_Tuning/H04_Well_lr001_batch16/weights/best.pt"

print("H03 exists:",os.path.exists(h03))
print("H04 exists:",os.path.exists(h04))

H03 exists: False
H04 exists: False


In [ ]:
import os

H03_YAML="/content/drive/MyDrive/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml"

print("H03 YAML exists:",os.path.exists(H03_YAML))
print("H03 YAML path:",H03_YAML)

H03 YAML exists: False
H03 YAML path: /content/drive/MyDrive/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H01.yaml


In [ ]:
import os

base="/content/drive/MyDrive/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8"

print("Folder exists:",os.path.exists(base))

if os.path.exists(base):
    print("Contents:")
    print(os.listdir(base))

Folder exists: True
Contents:
['train', 'data.yaml', 'valid', 'test']


In [ ]:
H03_LOCAL_YAML="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H03.yaml"
with open(H03_LOCAL_YAML,"w") as f:
    f.write("""path: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant
""")

print("H03 local YAML created:",os.path.exists(H03_LOCAL_YAML))

H03 local YAML created: True


In [ ]:
import os

print("Exists:",os.path.exists(H03_YAML))

with open(H03_YAML,"r") as f:
    print(f.read())

Exists: True
path: /content/drive/MyDrive/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant



In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 10.1 MB/s eta 0:00:00


In [ ]:
import os
import shutil
import time
datasets=[
    "Aquatic Plant.v2i.yolov8",
    "well.v8i.yolov8"
]
for dataset in datasets:
    source=f"/content/drive/MyDrive/Dataset_V1_Enhanced/{dataset}"
    destination=f"/content/Dataset_V1_Enhanced/{dataset}"
    print(f"\nChecking: {dataset}")
    if os.path.exists(destination):
        print("Already exists in Colab local storage.")
    else:
        print("Copying...")
        start=time.time()
        shutil.copytree(source,destination)
        elapsed=(time.time()-start)/60
        print(f"Copied in {elapsed:.2f} minutes.")
    print("Local folder exists:",os.path.exists(destination))
print("\nBoth datasets are ready in Colab local storage.")


Checking: Aquatic Plant.v2i.yolov8
Already exists in Colab local storage.
Local folder exists: True

Checking: well.v8i.yolov8
Copying...
Copied in 3.30 minutes.
Local folder exists: True

Both datasets are ready in Colab local storage.


In [ ]:
print(open(H03_LOCAL_YAML).read())

path: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant



In [ ]:
from ultralytics import YOLO
import time
H03_YAML="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H03.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H03_YAML,
    epochs=30,
    imgsz=640,
    batch=32,
    lr0=0.0005,
    optimizer="AdamW",
    device=0,
    project="/content/drive/MyDrive/Day22_Final_Model/H03_Training",
    name="Aquatic_Plant_H03",
    exist_ok=True
)
training_time=time.time()-start
print("\nH03 training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H03.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Aquatic_Plan

In [ ]:
H04_YAML="/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml"
with open(H04_YAML,"w") as f:
    f.write("""path: /content/Dataset_V1_Enhanced/well.v8i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 4
names:
  0: Inlet-pipe
  1: fishes
  2: school-of-fish
  3: stone
""")
print("H04 YAML created:",H04_YAML)

H04 YAML created: /content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml


In [ ]:
from ultralytics import YOLO
import time
H04_YAML="/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml"
model=YOLO("yolov8n.pt")
start=time.time()
results=model.train(
    data=H04_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.001,
    optimizer="AdamW",
    device=0,
    project="/content/drive/MyDrive/Day22_Final_Model/H04_Training",
    name="Well_H04",
    exist_ok=True
)
training_time=time.time()-start
print("\nH04 training completed.")
print(f"Training time: {training_time/60:.2f} minutes")

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/well.v8i.yolov8/H04.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Well_H04, nbs=64, nms=

In [ ]:
import os
paths={
    "H03_Aquatic_Plant":"/content/drive/MyDrive/Day22_Final_Model/H03_Training/Aquatic_Plant_H03/weights/best.pt",
    "H04_Well":"/content/drive/MyDrive/Day22_Final_Model/H04_Training/Well_H04/weights/best.pt",
    "DIAT_Baseline":"/content/drive/MyDrive/Day16_YOLO_Baseline/DIATAquarium/weights/best.pt"
}
for name,path in paths.items():
    print(name)
    print("Exists:",os.path.exists(path))
    if os.path.exists(path):
        print("Size:",round(os.path.getsize(path)/1024/1024,2),"MB")
    print()

H03_Aquatic_Plant
Exists: True
Size: 5.96 MB

H04_Well
Exists: True
Size: 5.96 MB

DIAT_Baseline
Exists: True
Size: 5.95 MB

